# 04 · Stability under acquisition shift

The question the repository exists to ask: when the acquisition changes and **the prediction does
not**, does the explanation change?

Comparing maps only on images whose predicted class is unchanged is what keeps the two effects
apart. Without that restriction, a fall in agreement could simply mean the model changed its mind.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, str(Path.cwd().parent / "src"))  # works without installing the package

CONFIG = "../configs/default.yaml"   # switch to ../configs/smoke.yaml to run offline in seconds


In [ ]:
from ctxaiqc.run_experiment import run
from ctxaiqc.utils import load_config

cfg = load_config(CONFIG)
tables = Path(cfg["experiment"]["out_dir"]) / "tables"

# Equivalent to `make experiment`; skipped if the tables are already there.
if not (tables / "saliency_stability.csv").exists():
    run(cfg)

In [ ]:
import pandas as pd

perf = pd.read_csv(tables / "performance_under_shift.csv")
stab = pd.read_csv(tables / "saliency_stability.csv")
perf[perf["perturbation"] == "dose"].round(3)

## The main figure\n\nPrediction on the left axis, explanation on the right, same images, same x axis.

In [ ]:
from ctxaiqc.report import family_figure, main_figure, summary_table

figures = Path(cfg["experiment"]["out_dir"]) / "figures"
main_figure(tables, figures / "stability_vs_dose.png", family="dose")
family_figure(tables, figures / "stability_by_family.png")
summary_table(tables, tables / "summary.md")

from IPython.display import Image, display

display(Image(str(figures / "stability_vs_dose.png")))
display(Image(str(figures / "stability_by_family.png")))

## The same thing, one image at a time

The aggregate curve says agreement falls. This shows what that looks like on a single image whose
predicted class did not change.

In [ ]:
from ctxaiqc.data import load_dataset
from ctxaiqc.evaluate import load_checkpoint, predict_probs
from ctxaiqc.explain import explain
from ctxaiqc.metrics import saliency_agreement
from ctxaiqc.perturb import apply_perturbation, levels

data = load_dataset(**cfg["dataset"], seed=cfg["seed"])
model, _ = load_checkpoint(cfg["train"]["checkpoint"])
raw = data.x_test_raw[0, 0]

doses = list(levels("dose"))
base_img = apply_perturbation(raw, "dose", doses[0], seed=0)
base_pred = int(predict_probs(model, base_img[None, None]).argmax())
base_sal = explain("gradcam", model, base_img[None], target=base_pred)

fig, axes = plt.subplots(2, len(doses), figsize=(2.2 * len(doses), 4.8))
for col, d in enumerate(doses):
    img = apply_perturbation(raw, "dose", d, seed=11)
    p = predict_probs(model, img[None, None])
    sal = explain("gradcam", model, img[None], target=base_pred)
    scores = saliency_agreement(base_sal, sal)
    axes[0, col].imshow(img, cmap="gray", vmin=0, vmax=1); axes[0, col].axis("off")
    axes[0, col].set_title(f"{d:g}x dose\npred {int(p.argmax())}", fontsize=8)
    axes[1, col].imshow(img, cmap="gray", vmin=0, vmax=1)
    axes[1, col].imshow(sal, cmap="inferno", alpha=0.55); axes[1, col].axis("off")
    axes[1, col].set_title(rf"$\rho$ = {scores['spearman']:.2f}", fontsize=8)
fig.tight_layout()

## What this does and does not license you to say

It licenses: *on this benchmark, under this simulator, the agreement of these saliency methods with
their own reference map falls while the prediction is unchanged.*

It does not license any claim about clinical CT, about a specific scanner, or about how these
methods behave across real institutions. The limits are set out in the README under **Scope and
limitations**, and the next step — generative counterfactuals instead of post-hoc saliency — is on
the `roadmap` branch.